# Smart Campus energy forecasting — training notebook (Track 1)

Full pipeline: problem understanding, EDA, cleaning, features, baselines, model comparison, evaluation, error analysis, final training, and saving `model.pkl`. The **prediction notebook** (uploaded to the platform) only loads `model.pkl` and predicts.

**Problem.** Predict hourly `energy_usage` of a campus building from building, time (hour, weekday, month), weather, occupancy and previous-hour usage. Metric: RMSE (MAE, R² also reported). Some inputs are missing.

**Key insights that drive the design**
1. Usage is close to *linear and additive per building*: a carry-over of ~0.35 x previous-hour usage, a type-specific daily profile, per-building slopes for occupancy and temperature, and a steeper cooling effect above ~29-31 °C.
2. `test.csv` comes from a **wider distribution** than train (heavier tails in temperature, humidity, occupancy, previous usage; more heat-wave / storm / transition-hour rows). Trees cannot extrapolate; a well-specified linear model can.
3. Test has **far more missing inputs** (18% of rows, up to 3 per row, vs 6% in train) and test's feature distribution is shifted, so missing values are imputed per availability pattern with models that also see test's (unlabeled) features.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, itertools, lightgbm as lgb
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
train = pd.read_csv('train.csv'); test = pd.read_csv('test.csv')
print(train.shape, test.shape); print(train.head().to_string())

## 1. Exploratory data analysis

In [ ]:
NUMC=['temperature','humidity','occupancy','previous_usage']
print('missing per column (train/test):'); print(pd.DataFrame({'train':train[NUMC].isna().mean(),'test':test[NUMC].isna().mean()}).round(3))
print('rows with >=1 missing: train %.3f, test %.3f' % (train[NUMC].isna().any(axis=1).mean(), test[NUMC].isna().any(axis=1).mean()))
fig,ax=plt.subplots(1,4,figsize=(18,3.5))
for a,c in zip(ax,NUMC):
    a.hist(train[c].dropna(),bins=40,density=True,alpha=.55,label='train'); a.hist(test[c].dropna(),bins=40,density=True,alpha=.55,label='test'); a.set_title(c); a.legend()
plt.suptitle('Train vs test feature distributions: test has heavier tails / extra edge-case clusters'); plt.show()

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(18,4))
ax[0].scatter(train.previous_usage,train.energy_usage,s=3,alpha=.3); ax[0].plot([0,150],[0,150],'r--'); ax[0].set_xlabel('previous_usage'); ax[0].set_ylabel('energy_usage'); ax[0].set_title('corr %.2f'%train[['previous_usage','energy_usage']].corr().iloc[0,1])
for t,g in train.groupby('building_type'): ax[1].plot(g.groupby('hour').energy_usage.mean(),label=t)
ax[1].legend(fontsize=7,ncol=2); ax[1].set_title('Daily profile by building type'); ax[1].set_xlabel('hour')
d=train.dropna(); r=d.energy_usage-0.35*d.previous_usage; b=pd.cut(d.temperature,np.arange(23,36.5,1))
ax[2].plot([i.mid for i in r.groupby(b,observed=True).mean().index],r.groupby(b,observed=True).mean().values,'o-'); ax[2].set_title('Temperature effect (after prev carry-over): steeper when hot'); ax[2].set_xlabel('temperature')
plt.show()

## 2. Cleaning and feature engineering
No impossible values were found (humidity <= 100, occupancy >= 0, previous usage >= 0, building type consistent with building id), so no rows are dropped; the rules stay in `clean()` as guards. `design()` builds the structural design matrix.

In [ ]:
import numpy as np, pandas as pd
DAYS=['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
BLD=['ADM_A','BUS_A','BUS_B','ENG_A','ENG_B','LEC_A','LIB_A','RES_A','RES_B','SCI_A','SCI_B','SPT_A']
TYP=['Administration','Business','Engineering','LectureHall','Library','Residential','Science','Sports']
B2T={'ADM_A':'Administration','BUS_A':'Business','BUS_B':'Business','ENG_A':'Engineering','ENG_B':'Engineering',
     'LEC_A':'LectureHall','LIB_A':'Library','RES_A':'Residential','RES_B':'Residential','SCI_A':'Science','SCI_B':'Science','SPT_A':'Sports'}
NUMS={'t':'temperature','h':'humidity','o':'occupancy','p':'previous_usage'}
FULL=('t','h','o','p')

def clean(df):
    """Step 4 rules (no rows dropped): types, clip impossible values, derive dow/type."""
    df=df.copy()
    for c in NUMS.values(): df[c]=pd.to_numeric(df[c],errors='coerce')
    df['humidity']=df.humidity.clip(0,100); df['occupancy']=df.occupancy.clip(lower=0)
    df.loc[df.previous_usage<0,'previous_usage']=np.nan
    df['dow']=df.day_of_week.map({d:i for i,d in enumerate(DAYS)}).fillna(0).astype(int)
    df['building_type']=df.building_id.map(B2T).fillna(df.get('building_type'))
    df['hour']=df.hour.astype(int)%24; df['month']=df.month.astype(int)
    return df

def _oh(v,cats):
    v=np.asarray(v); return (v[:,None]==np.asarray(cats)[None,:]).astype(float)

def design(df,avail=FULL,hinge=29.0):
    """Structural design matrix: building/hour/dow/month effects, type x hour profile,
    type x weekend, per-building slopes for the available numeric features, cooling hinge."""
    B=_oh(df.building_id,BLD); T=_oh(df.building_type,TYP)
    H=_oh(df.hour,range(24)); W=_oh(df.dow,range(7)); M=_oh(df.month,range(1,13))
    we=(df.dow.values>=5).astype(float)[:,None]
    x=lambda A,G:(G[:,:,None]*A[:,None,:]).reshape(len(df),-1)
    parts=[B,H,W,M,x(H,T),x(we,T)]
    cols=[]
    if 't' in avail: t=df.temperature.values; cols.append(t-28)
    if 'o' in avail: cols.append(df.occupancy.values/100)
    if 'p' in avail: cols.append(df.previous_usage.values/50)
    if cols:
        num=np.column_stack(cols); parts+=[num,x(num,B)]
    if 'h' in avail:
        hu=(df.humidity.values[:,None]-78)/5; parts+=[hu,x(hu,T)]
    if 't' in avail and hinge is not None: parts.append(np.maximum(t-hinge,0)[:,None])
    return np.hstack(parts)

def gbm_frame(df,avail):
    X=pd.DataFrame({'hour':df.hour.values,'dow':df.dow.values,'month':df.month.values,
                    'b':pd.Categorical(df.building_id,BLD)})
    for a in avail: X[NUMS[a]]=df[NUMS[a]].values
    return X

# ---- v8 design: group-scaled blocks (scale = inverse penalty strength), tuned on (adversarially weighted) CV ----
SCALE_V8={"B": 2.0, "H": 0.0625, "W": 1.0, "M": 0.25, "TH": 0.25, "Twe": 1.0, "num": 2.0, "Bnum": 1.0, "hu": 1.0, "huT": 0.03125, "hinge": 1.0, "Bwe": 0.08, "o_hb_T": 0.04, "p_hb_T": 0.32, "t_hb": 0.04}
def design_v8(df,hinge=29.0):
    B=_oh(df.building_id,BLD); T=_oh(df.building_type,TYP); H=_oh(df.hour,range(24)); W=_oh(df.dow,range(7)); M=_oh(df.month,range(1,13))
    we=(df.dow.values>=5).astype(float)[:,None]; x=lambda A,G:(G[:,:,None]*A[:,None,:]).reshape(len(df),-1)
    t=df.temperature.values; o=df.occupancy.values/100; p=df.previous_usage.values/50
    num=np.column_stack([t-28,o,p]); hu=(df.humidity.values[:,None]-78)/5
    hb=np.column_stack([(df.hour.values>=a)&(df.hour.values<a+4) for a in range(0,24,4)]).astype(float)
    bl={'B':B,'H':H,'W':W,'M':M,'TH':x(H,T),'Twe':x(we,T),'num':num,'Bnum':x(num,B),'hu':hu,'huT':x(hu,T),'hinge':np.maximum(t-hinge,0)[:,None],
        'Bwe':x(we,B),'o_hb_T':x(o[:,None]*hb,T),'p_hb_T':x(p[:,None]*hb,T),'t_hb':(t[:,None]-28)*hb,'o2_T':x((o**2)[:,None],T)}
    return np.hstack([bl[k]*SCALE_V8[k] for k in SCALE_V8])

## 3. Validation design
5-fold CV, plus **extrapolation folds** (train on the middle of a feature's range, score on its tails) and **test-like missingness** checks.

In [ ]:
tr=clean(train); te=clean(test)
d=tr.dropna(subset=NUMC).reset_index(drop=True); y=d.energy_usage.values
def rmse(a,b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
folds=list(KFold(5,shuffle=True,random_state=0).split(d))
def cv(fitpred):
    P=np.zeros(len(d))
    for a,b in folds: P[b]=fitpred(d.iloc[a],y[a],d.iloc[b])
    return P
def ext_fold(fitpred,col,lo,hi):
    ql,qh=d[col].quantile([lo,hi]); t=((d[col]<ql)|(d[col]>qh)).values if lo>0 else (d[col]>qh).values
    return rmse(fitpred(d[~t],y[~t],d[t]),y[t])

## 4. Baselines and model comparison

In [ ]:
def plainX(df):
    X=pd.DataFrame({f'b_{b}':(df.building_id==b).astype(float) for b in BLD})
    for c in ['hour','dow','month']+NUMC: X[c]=df[c].values
    return X
models={
 'mean dummy': lambda A,ya,B: np.full(len(B),ya.mean()),
 'plain linear': lambda A,ya,B: LinearRegression().fit(plainX(A),ya).predict(plainX(B)),
 'LightGBM': lambda A,ya,B: lgb.LGBMRegressor(n_estimators=1600,learning_rate=0.02,num_leaves=15,subsample=0.8,subsample_freq=1,colsample_bytree=0.8,verbose=-1).fit(gbm_frame(A,FULL),ya).predict(gbm_frame(B,FULL)),
 'structural ridge': lambda A,ya,B: Ridge(1.0).fit(design(A),ya).predict(design(B)),
}
table={}
for name,fn in models.items():
    P=cv(fn)
    entry={'CV RMSE':rmse(P,y),'MAE':mean_absolute_error(y,P),'R2':r2_score(y,P)}
    for col,lo_q,hi_q,label in [('temperature',.06,.94,'temp tails'),('occupancy',0,.9,'occupancy tail'),('previous_usage',.06,.94,'prev tails')]:
        entry[label]=ext_fold(fn,col,lo_q,hi_q)
    table[name]=entry
results=pd.DataFrame.from_dict(table,orient='index'); print(results.round(3).to_string())

The structural ridge wins on random folds **and** stays accurate on the extrapolation folds where LightGBM degrades sharply (it flattens outside the training range). Boosting on ridge residuals, linear-tree LightGBM, splines and 20+ interaction variants were also tried (see `ml_pipeline/runs.jsonl`): none improved, i.e. the remaining error (~3.0) is noise.

## 5. Missing inputs
For each availability pattern, the missing inputs are imputed by LightGBM trained on the available inputs (train rows **plus unlabeled test features**, weight 10, LightGBM+XGBoost average — test's feature distribution is shifted, e.g. occupancy imputed from train alone is biased by -5.9 on test's own observed values), then the full ridge is applied. Rows missing both occupancy and previous usage use the average of that and a NaN-native LightGBM. Small additive offsets per missing feature are learned out-of-fold on train's real missing rows.

In [ ]:
import itertools, numpy as np, pandas as pd, lightgbm as lgb, xgboost as xgb
from sklearn.linear_model import Ridge
IMP_GP=dict(n_estimators=400,learning_rate=0.04,num_leaves=31,min_child_samples=20,subsample=0.8,subsample_freq=1,verbose=-1,n_jobs=2,random_state=0)
TREE_GP=dict(n_estimators=1600,learning_rate=0.02,num_leaves=15,subsample=0.8,subsample_freq=1,colsample_bytree=0.8,verbose=-1,n_jobs=2,random_state=0)
OFFSETS={'t':-0.27,'h':0.18,'o':0.33,'p':0.79}
def xgb_frame(df,av):
    X=gbm_frame(df,av).copy(); X['b']=X.b.cat.codes; return X
def impute(pair,df,av):
    return 0.5*(pair[0].predict(gbm_frame(df,av))+pair[1].predict(xgb_frame(df,av)))

def build(df,unlabeled=None,test_weight=10.0,alpha=1.0,hinge=29.0,offsets=OFFSETS,c_mode='tree'):
    df=clean(df); d=df.dropna(subset=list(NUMS.values())).reset_index(drop=True)
    main=Ridge(alpha).fit(design(d,FULL,hinge),d.energy_usage.values)
    main8=Ridge(alpha).fit(design_v8(d,hinge),d.energy_usage.values)
    feat=df.drop(columns=['energy_usage']).assign(_w=1.0)
    if unlabeled is not None:
        u=clean(unlabeled).assign(_w=test_weight); feat=pd.concat([feat,u[feat.columns]],ignore_index=True)
    imps={}
    for k in range(1,5):
        for ms in itertools.combinations(FULL,k):
            av=tuple(a for a in FULL if a not in ms)
            for mv in ms:
                sub=feat.dropna(subset=[NUMS[mv]]+[NUMS[a] for a in av])
                g1=lgb.LGBMRegressor(**IMP_GP).fit(gbm_frame(sub,av),sub[NUMS[mv]].values,sample_weight=sub._w.values)
                g2=xgb.XGBRegressor(n_estimators=500,learning_rate=0.04,max_depth=6,subsample=0.8,colsample_bytree=0.9,n_jobs=2,random_state=0).fit(xgb_frame(sub,av),sub[NUMS[mv]].values,sample_weight=sub._w.values)
                imps[(mv,av)]=(g1,g2)
    tree=lgb.LGBMRegressor(**TREE_GP).fit(gbm_frame(df,FULL),df.energy_usage.values)
    return {'main':main,'main8':main8,'imputers':imps,'tree':tree,'hinge':hinge,'offsets':dict(offsets),'c_mode':c_mode,'version':'ridge-struct-v6'}
def predict(art,df,c_mode=None,use_offsets=True):
    c_mode=c_mode or art['c_mode']
    df=clean(df).reset_index(drop=True); out=np.zeros(len(df))
    miss=df[list(NUMS.values())].isna().values; keys=[tuple(r) for r in miss]
    for pat in set(keys):
        rows=np.array([i for i,k in enumerate(keys) if k==pat]); sub=df.iloc[rows].copy()
        ms=tuple(a for a,m in zip(FULL,pat) if m); av=tuple(a for a in FULL if a not in ms)
        for mv in ms: sub[NUMS[mv]]=impute(art['imputers'][(mv,av)],df.iloc[rows],av)
        p=0.5*(art['main'].predict(design(sub,FULL,art['hinge']))+art['main8'].predict(design_v8(sub,art['hinge'])))
        if use_offsets: p=p+sum(art['offsets'][m] for m in ms)
        if 'o' in ms and 'p' in ms and c_mode!='ridge':
            t=art['tree'].predict(gbm_frame(df.iloc[rows],FULL))
            p= t if c_mode=='tree' else 0.5*(p+t)
        out[rows]=p
    return out

## 6. Final training, error analysis and saving the model

In [ ]:
oof=np.zeros(len(train))
for a,b in KFold(5,shuffle=True,random_state=0).split(train):
    m=build(train.iloc[a],unlabeled=test,c_mode='blend'); oof[b]=predict(m,train.iloc[b])
yt=train.energy_usage.values; miss=train[NUMC].isna().any(axis=1).values
print('OOF on all train rows: RMSE %.3f  MAE %.3f  R2 %.4f' % (rmse(oof,yt),mean_absolute_error(yt,oof),r2_score(yt,oof)))
print('complete rows RMSE %.3f | rows with missing inputs RMSE %.3f' % (rmse(oof[~miss],yt[~miss]),rmse(oof[miss],yt[miss])))
r=yt-oof; fig,ax=plt.subplots(1,2,figsize=(14,4))
ax[0].scatter(oof,r,s=3,alpha=.3); ax[0].axhline(0,color='r'); ax[0].set_xlabel('prediction'); ax[0].set_ylabel('residual'); ax[0].set_title('Residuals: centred, spread grows with usage')
pd.Series(r).groupby(train.building_id).apply(lambda s: np.sqrt((s**2).mean())).plot.bar(ax=ax[1]); ax[1].set_title('RMSE by building'); plt.show()

In [ ]:
final=build(train,unlabeled=test,c_mode='blend')
joblib.dump(final,'model.pkl',compress=('xz',9))
preds=predict(final,test)
sub=pd.DataFrame({'id':test['id'].values,'prediction':preds}); sub.to_csv('submission.csv',index=False); print(sub.head().to_string())

## 7. Assumptions, limitations
* **Assumptions:** effects are additive and linear within building (validated: residual boosting finds no structure); noise is roughly proportional to usage; the test-generating process for complete rows matches train's; previous_usage is the true previous-hour reading (legitimately available at prediction time).
* **Limitations:** rows with several missing inputs are much harder (no train analogue); imputation uses unlabeled test features (transductive) — if the private set's feature distribution differs, imputers fall back to train+test statistics; irreducible noise ~3.0 RMSE.
* **With more time/data:** timestamps would allow true lag features; sensor-failure flags would explain test's structured missingness.